In [1]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI


c:\Users\dev\Documents\1-FY2026-VA-Course\1-va-intermediate\rag-projects\rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [5]:
knowledge = {}
filenames = glob.glob("knowledge-base/employees/*")
for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()



In [ ]:
knowledge["lancaster"]


In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()


In [ ]:
knowledge.keys()


dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

In [10]:
SYSTEM_PREFIX = """
You are Insurellm AI, an intelligent assistant for Insurellm, an Insurance Technology company.

Your role is to help employees by answering questions about:
- Insurellm products and services
- Company policies and procedures
- Employee-related information
- Insurance-related knowledge contained in the provided documents

Instructions:
1. Use the provided context as your primary source of information.
2. Answer clearly, accurately, and concisely.
3. If the answer is not available in the provided context, say:
   "I don't have enough information to answer that question."
4. Do not make up facts or assumptions.
5. When possible, base your response directly on the retrieved information.

Relevant Context:
"""


In [11]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   


In [ ]:
get_relevant_context("Who is lancaster?")


In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result


In [ ]:
print(additional_context("Who is Alex Lancaster?"))


The following additional context might be relevant in answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joini

In [20]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [22]:
view = gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
